In [1]:
import random
import json
from pathlib import Path
from enum import Enum
from typing import Tuple, List, Optional, Callable, Union, Literal

import torch
from torch.utils.data import Dataset
from torchvision.transforms import v2

In [3]:
class SplitType(Enum):
    TRAIN = 'train'
    VAL = 'val'
    TEST ='test'

class Sentinel(Dataset):
    def __init__(self, root_dir:Union[str, Path], split_type: Optional[str]=None, transform: Optional[Callable]= None, split_mode: Literal['random', 'split']='random', split_ratio: Tuple[float, float, float]=(0.7, 0.15, 0.15), split_file: Optional[Union[str, Path]]=None, seed: int=42):
        self.root_dir =Path(root_dir)
        if not self.root_dir.exists():
            raise FileNotFoundError(f"Dataset root dir not found {self.root_dir}")
        
        #convert string split_type to enum
        self.split_type = SplitType(split_type) if split_type else None

        #default transform pipeline
        self.transform = transform if transform else v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True)
        ])

        #collecting image pairs
        self.all_image_pairs = self._collect_images()

        #apply split if specified
        if split_type:
            if split_mode =='split' and split_file:
                self.image_pairs =self._apply_predefined_split(split_file)
            elif split_mode == 'random':
                self.image_pairs = self._apply_random_split(split_ratio, seed)
            else:
                raise ValueError("Invalid split config")
        else:
            # if no split type specified use all images
            self.image_pairs = self.all_image_pairs
        
        print(f'Total image pairs found: {len(self)}')

    def _collect_images(self) -> List[Tuple[Path, Path]]:
        #collected pairs s1 and optical s2 image path from dir
        image_pairs = []

        for category in self.root_dir.iterdir():
            if not category.is_dir():
                continue

            s1_path = category / 's1'
            s2_path =category / 's2'

            if not (s1_path.is_dir() and s2_path.is_dir()):
                continue

            #collect pairs
            for s1_file in s1_path.glob('*.png'):
                s2_filename = list(s1_file.name.split('_'))
                s2_filename[2] = 's2'
                s2_file =s2_path / '_'.join(s2_filename)

                if not s2_file.exists():
                    continue

                image_pairs.append((s1_file, s2_file))
            return image_pairs
        
        def predefined_split(self, split_file:Union[str, Path])-> List[Tuple[Path, Path]]:
            try:
                with open(split_file,'r') as f:
                    splits = json.load(f)
                
                if self.split_type.value not in splits['data']:
                    raise ValueError(f"Split type {self.split_type.value} not found in split file")
            
                split_filenames = set(splits['data'][self.split_type.value]) # data['split']
                return [pair for pair in self.all_image_pairs 
                    if any(p.name in split_filenames for p in pair[:2])]
            
            except Exception as e:
                print(f'Could not open split file\n\t{e}')
                raise
        
        def apply_random_split(self, split_ratio: Tuple[float, float, float], seed: int)-> List[Tuple[Path, Path]]:
            if sum(split_ratio) != 1:
                raise ValueError("Split ratios must sum to 1")
        
            # set random seed for reproducibility
            random.seed(seed)
        
            # shuffle indices
            indices = list(range(len(self.all_image_pairs)))
            random.shuffle(indices)
        
            # calculate split points
            train_end = int(len(indices) * split_ratio[0])
            val_end = train_end + int(len(indices) * split_ratio[1])
        
            # select appropriate slice based on split type
            if self.split_type == SplitType.TRAIN:
                split_indices = indices[:train_end]
            elif self.split_type == SplitType.VAL:
                split_indices = indices[train_end:val_end]
            else:  # TEST
                split_indices = indices[val_end:]
            
            return [self.all_image_pairs[i] for i in split_indices]
        
        def save_split(self, output_file: Union[str, Path], append:bool =False):
            if self.split_type:
                split = self.split_type.value
                split_info = {
                    'data': {
                        split: [P[0].name for p in self.image_pairs]
                    }
                }

                mode= 'a' if append else 'w'
                with open(output_file, mode) as f:
                    json.dump(split_info, f, indent=2)
        
        def __len__(self):
            return len(self.image_pairs)
        
        def __getitem__(self, idx:int)-> Tuple[torch.Tensor, torch.Tensor]:
            s1_path, s2_path =self.image_pairs[idx]

            #load images
            s1_image = Image.open(s1_path).convert('RGB')
            s2_image = Image.open(s2_path).convert('RGB')

            #applly transform
            s1_image = self.transform(s1_image)
            s2_image = self.transform(s2_image)
        
            return s1_image, s2_image
            
            
            

## PIX2PIX Model implementation

In [4]:
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class Downsample(nn.Module): #it consists of convolution batchnorm relu layer with k filter
    def __init__(self, c_in, c_out, kernel_size=4, stride=2, padding=1, negative_slope=0.02, use_norm=True):
        #intialzies the unet downsample block
        # c_in (int): The number of input channels.
        # c_out (int): The number of output channels.
        # kernel_size (int, optional): The size of the convolving kernel. Default is 4.
        # stride (int, optional): Stride of the convolution. Default is 2.
        # padding (int, optional): Zero-padding added to both sides of the input. Default is 0.
        # negative_slope (float, optional): Negative slope for the LeakyReLU activation function. Default is 0.2.
        # use_norm (bool, optinal): If use norm layer. If True add a BatchNorm layer after Conv. Default is True.

        super(Downsample, self).__init__()
        block =[]
        block += [nn.Convo2d(in_channel=c_in, out_channel=c_out, kernel_size=kernel_size, sttride= stride, padding=padding, bias=(not use_norm))]

        if use_norm:
            block += [nn.BatchNoemd(num_features=c_out)]
        
        block += [nn.LeakyReLU(negative_slope = negative_slope)]

        self.conv_block = nn.Sequential(*block)

    
    def forward(self,x) :
        return self.conv_block(x)
    

In [ ]:
class Upsampling(nn.Module): #unet upsampling block

    def __init__(self, c_in, c_out, kernel_size=4, stride=2,padding=1, use_dropout=False, use_upsampling=False, mode='nearest'):
        # use_dropout (bool, optional): if use dropout layers. Default is False.
        # upsample (bool, optinal): if use upsampling rather than transpose convolution. Default is False.
        # mode (str, optional): the upsampling algorithm: one of 'nearest','bilinear', 'bicubic'. Default: 'nearest'
        
        super(Upsampling, self).__init__()
        block= []
        if use_upsampling:
            # transpose convolutional cause checkerboard artifacts.upsampling
            # followed by regular convolutional , produces better result apperantly

            mode= mode if mode in('nearest', 'bilinear','bicubic')else 'nearest'
            block += [nn.Sequential(nn.Upsample(scale_factor=2, mode=mode), nn.Conv2d(in_channels=c_in, out_channels=c_out, kernel_size=3, stride=1, padding=padding, bias=False))]

        else:
            block += [nn.ConvTranspose2d(in_channels=c_in, out_channels=c_out, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)]

            block += [nn.BatchNorm2d(num_features=c_out)]

            if use_dropout:
                block += [nn.Dropout(0.5)]
            
            block += [nn.ReLU()]
            self.conv_block = nn.Sequential(*block)
        
    def forward(self, x):
        return self.conv_block(x)


In [ ]:
class UnetEncoder(nn.Module):
    #unet 

    def __init__(self, c_in=3, c_out=512): #unet encoder network

        super(UnetEncoder, self).__init()
        self.enc1 = Downsample(c_in, 64, use_norm=False) #c64
        self.enc2 = Downsample(64, 128) #c128
        self.enc3 = Downsample(128, 256) #c256
        self.enc4 = Downsample(256, 512) #c512
        self.enc5 = Downsample(512, 512) #c512
        self.enc6 = Downsample(512, 512) #c512
        self.enc7 = Downsample(512, 512) #c512
        self.enc8 = Downsample(512, c_out) #c512
        
        def forward(self, x):
            x1 = self.enc1(x)
            x2 = self.enc2(x1)
            x3 = self.enc3(x2)
            x4 = self.enc4(x3)
            x5 = self.enc5(x4)
            x6 = self.enc6(x5)
            x7 = self.enc7(x6)
            x8 = self.enc8(x7)
            out = [x8, x7, x6, x5, x4, x3, x2, x1] # latest activation is the first element
            return out

In [ ]:
class UnetDecoder(nn.Module): #unet decoder network

    #ck -convolutional batchnorm relu layer with k filter
    #cdk - convolutional batchnorm dropout relu layer with dropout rate of 50%
    def __init__(self, c_in=512, c_out=64, use_upsampling=False, mode='nearest'):
        super(UnetDecoder, self).__init__()

        self.dec1 = Upsampling(c_in, 512, use_dropout=True, use_upsampling=use_upsampling, mode=mode) #cd512
        self.dec2 = Upsampling(1024, 512, use_dropout=True, use_upsampling=use_upsampling, mode=mode) #cd1024
        self.dec3 = Upsampling(1024, 512, use_dropout=True, use_upsampling=use_upsampling, mode=mode) #cd1024
        self.dec4 = Upsampling(1024, 512, use_upsampling=use_upsampling, mode=mode) #cd1024
        self.dec5 = Upsampling(1024, 256, use_upsampling=use_upsampling, mode=mode) #cd1024
        self.dec6 = Upsampling(512, 128, use_upsampling=use_upsampling, mode=mode) #cd512
        self.dec7 = Upsampling(256, 64, use_upsampling=use_upsampling, mode=mode) #cd256
        self.dec8 = Upsampling(128, c_out, use_upsampling=use_upsampling, mode=mode) #cd128

        def forward(self, x):
            x9 = torch.cat([x[1], self.dec1(x[0])], 1) # (N,1024,H,W)
            x10 = torch.cat([x[2], self.dec2(x9)], 1) # (N,1024,H,W)
            x11 = torch.cat([x[3], self.dec3(x10)], 1) # (N,1024,H,W)
            x12 = torch.cat([x[4], self.dec4(x11)], 1) # (N,1024,H,W)
            x13 = torch.cat([x[5], self.dec5(x12)], 1) # (N,512,H,W)
            x14 = torch.cat([x[6], self.dec6(x13)], 1) # (N,256,H,W)
            x15 = torch.cat([x[7], self.dec7(x14)], 1) # (N,128,H,W)
            out = self.dec8(x15) # (N,64,H,W)
            return out




In [ ]:
class UnetGenerator(nn.Module):
    #unet based generator
    def __init__(self, c_in=3, c_out=3, use_upsampling=False, mode= 'nearest'):

        super(UnetGenerator,self).__init__()
        self.encoder = UnetEncoder(c_in=c_in)
        self.decoder = UnetDecoder(use_upsampling=use_upsampling, mode=mode)

        #in Lua implementation, only tanh layer is applied instead following the original pix2pix imp(after last layer in decoder , a convolutional is applied to map to the number of output channel ,followed by tanh func)
        self.head =nn.Sequential(nn.Conv2d(in_channels=64, out_channels=c_out, kernel_size=3, stride=1, padding=1, bias=True), nn.Tanh())

    def forward(self, x):
        outE = self.encoder(x)
        outD = self.decoder(outE)
        out = self.head(outD)
        return out

In [ ]:
class PixelDisc(nn.Module):
    # create a pixelGAN discriminator (1*1 patchGAN)
    def __init__(self, c_in=3, c_hid=64):
    # all convolutions are 1*1 saptial filters

    # c_in (int, optional): The number of input channels. Defaults to 3
    # c_hid (int, optional): The number of channels after first conv layer defaults to 64
        super(PixelDisc, self).__init__()
        self.model = nn.Sequential(Downsample(c_in, c_hid, kernel_size=1, stride=1, padding=0, use_norm=False),
            Downsample(c_hid, c_hid*2, kernel_size=1, stride=1, padding=0),
            nn.Conv2d(in_channels=c_hid*2, out_channels=1, kernel_size=1)
            )
        # similar to PatchDiscriminator, there should be a sigmoid layer at the end of discriminator.
        # however, nn.BCEWithLogitsLoss combines the sigmoid layer with BCE loss, providing greater numerical stability. Therefore, the discriminator outputs logits to take advantage of this stability.

    def forward(self, x):
        return self.model(x) 


In [ ]:
class PatchDisc(nn.Module):
    #patchGAN Discriminator
    def __init__(self,c_in=3, c_hid=64, n_layers=3):

        super(PatchDisc, self).__inint__()
        model = [Downsample(c_in, c_hid, use_norm=False)]
        n_p = 1 #multiplier for previous channel
        n_c = 1 #multiplier for current channel

        for n in range(1, n_layers):
            n_p = n_c
            n_c = min(2**n, 8) #the number channel is 512 at most

            model += [Downsample(c_hid*n_p, c_hid*n_c)]
        n_p = n_c
        n_c = min(2**n_layers, 8)
        model += [Downsample(c_hid*n_c,c_hid*n_c, stride=1)]
        #last layer of convolution followed by sigmoid func
        model += [nn.Conv2d(in_channels=c_hid*n_c, out_channels=1, kernel_size=4, stride=1, padding=1, bias=True)]

        self.model = nn.Sequential(*model)

    def forward(self,x):
        return self.model(x)




In [ ]:
#general patchGAN
class PatchGAN(nn.Module):
    def __init__(self, c_in=3, c_hid=64, mode='patch', n_layers=3):
        super(PatchGAN, self).__init__()
        if mode == 'pixel':
            self.model =PixelDisc(c_in, c_hid)
        else:
            self.model = PatchDisc(c_in, c_hid, n_layers)
        
    def forward(self, x):
        return self.model(x)


In [ ]:
class Pix2Pix(nn.Module):
    #Pix2Pix class , it is model for image to image translational task
    #by default model uses unet arch for generator with transposed convolution
    # the descriminator is 70*70 PatchGAN disciminator 

    def __init__(self, 
                 c_in: int = 3, 
                 c_out: int = 3, 
                 is_train: bool = True,
                 netD: str = 'patch',
                 lambda_L1: float = 100.0,
                 is_CGAN: bool= True,
                 use_upsampling: bool = False,
                 mode: str = 'nearest',
                 c_hid: int =64,
                 n_layers: int = 3,
                 lr: float = 0.0002,
                 beta1: float =0.5,
                 beta2: float = 0.999,
                 ):
        super(Pix2Pix, self).__init__()
        self.is_CGAN =is_CGAN
        self.lambda_L1 =lambda_L1

        self.gen = UnetGenerator(c_in=c_in, c_out=c_out, use_upsampling=use_upsampling, mode=mode)
        self.gen = self.gen.apply(self.weights_init)

        if is_train:
            # conditional GAN need both input and output together
            disc_in = c_in + c_out if is_CGAN else c_out
            self.disc = PatchGAN(c_in=disc_in, c_hid=c_hid, mode=netD, n_layers=n_layers) 
            self.disc = self.disc.apply(self.weights_init)

            #initialize optimizers
            self.gen_optimizer = torch.optim.Adam(
                self.gen.parameters(), lr=lr, betas=(beta1, beta2))
            self.disc_optimizer = torch.optim.Adam(
                self.disc.parameters(), lr=lr, betas=(beta1, beta2))

            # loss func
            self.criterion = nn.BCEWithLogitsLoss()
            self.criterion_L1 = nn.L1Loss()

    def forward(self, x):
        return self.gen(x)
    
    @staticmethod
    def weights_init(m):
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.constant_(m.bias, 0.0)
        if isinstance(m, nn.BatchNorm2d):
            nn.init.normal_(m.weight, 1.0, 0.02)
            nn.init.constant_(m.bias, 0)

    def disc_input(self, real_images, target_images, fake_images):
        # discriminator inputs based on conditional/unconditional setup
        if self.is_CGAN:
            real_AB =torch.cat([real_images, target_images], dim=1)
            fake_AB = torch.cat([real_images, 
                               fake_images.detach()], 
                               dim=1)
        else:
            real_AB = target_images
            fake_AB = fake_images.detach()
        return real_AB, fake_AB
    
    def gen_input(self, real_images, fake_images):
        if self.is_CGAN:
            fake_AB = torch.cat([real_images, 
                               fake_images], 
                               dim=1)
        else:
            fake_AB =fake_images
        return fake_AB
    
    def step_discriminator(self, real_images, target_images, fake_images):
        #discriminator forward/backward pass
        
        #prepare inputs
        real_AB, fake_AB =self._get_disc_inputs(real_images, target_images, 
                                                fake_images)
    
        pred_real =self.disc(real_AB) # D(x, y)
        pred_fake = self.disc(fake_AB) # D(x, G(x))

        # compute the losses
        lossD_real = self.criterion(pred_real, torch.ones_like(pred_real)) # (D(x, y), 1)
        lossD_fake = self.criterion(pred_fake, torch.zeros_like(pred_fake)) # (D(x, y), 0)
        lossD = (lossD_real + lossD_fake) * 0.5 # combined Loss
        return lossD
    
    def step_generator(self, real_images, target_images, fake_images):
        
        fake_AB = self._get_gen_inputs(real_images, fake_images)
          
        #forward pass through the discriminator
        pred_fake =self.disc(fake_AB)

        lossG_GaN =self.criterion(pred_fake, torch.ones_like(pred_fake)) # GAN Loss
        lossG_L1 =self.criterion_L1(fake_images, target_images)           # L1 Loss
        lossG = lossG_GaN + self.lambda_L1 * lossG_L1                      # Combined Loss
        #return total loss and individual components
        return lossG, {
            'loss_G': lossG.item(),
            'loss_G_GAN': lossG_GaN.item(),
            'loss_G_L1' : lossG_L1.item()
        }
    
    def train_step(self, real_images, target_images):
        #perform single trining step

        # forward pass through generator
        fake_images = self.forward(real_images)

        # update  discriminator
        self.disc_optimizer.zero_grad()
        lossD = self.step_discriminator(real_images, target_images, fake_images) #compute the loss
        lossD.backward()
        self.disc_optimizer.step()

        # update generator
        self.gen_optimizer.zero_grad() # reset the gradients for D
        lossG, G_losses = self.step_generator(real_images, target_images, fake_images) # compute the loss
        lossG.backward()
        self.gen_optimizer.step() #

        # return all losses
        return {
            'loss_D': lossD.item(),
            **G_losses

        }
    
    def current_visuals(self, real_images, target_images):
        # return visualization images.
        
        with torch.no_grad():
            fake_images = self.gen(real_images)
        return {
            'real': real_images,
            'fake': fake_images,
            'target': target_images
        }
        



In [ ]:
#hyperparameters
PARAMS = {
    'netD' : 'patch',
    'lambda_L1' : 100.0,
    'is_CGAN' :True,
    'use_upsampling' : False,
    'mode': 'nearest',
    'c_hid': 64,
    'n_layers' : 3,
    'lr' :0.0002,
    'beta1' : 0.5,
    'beta2' :0.999,
    'batch_size' : 32,
    'epochs' : 1,
    'seed' : 42
    }

SEED = PARAMS['seed']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)